# Task 3 - Step 1: Train DAN-DG and SAM (+ DAN-DG λ study)

Every run uses the Task 2 training loop (`shared/training.py`) with the identical protocol: initialisation,
domain-balanced batches (8 per source domain), crops/flips, AdamW (lr 1e-4, wd 1e-4), ≤ 30 epochs of 235 updates,
frozen BatchNorm statistics, early stopping and checkpoint selection on mean source-validation macro-F1.

| Config | Objective |
|---|---|
| `erm` | **not trained**: the Task 2 Source-only checkpoint |
| `dan_dg` | CE + (1/3)·Σ_{pairs} MK-MMD² between source domains (λ_DG = 1) |
| `sam` | SAM (ρ = 0.05) on CE; two forward/backward passes per batch |
| `dan_dg_lambda0.1`, `dan_dg_lambda10` | controlled study |

All methods have `uses_target = False`: the training loop refuses target images for them, and **no Sketch image is
loaded anywhere in this notebook**. Finished runs are skipped, never overwritten; `TASK3_RUNS=...` trains a subset.

In [1]:
# ---- Task 3 common header (identical in every Task 3 notebook) ----
# NOTE: this header never loads any Sketch image or label. Only notebook 03 does, after the freeze check.
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "shared" / "pacs.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared import pacs, pacs_protocol as proto
from shared.config import load_config

T3 = REPO / "task3"
CFG_DIR = T3 / "configs"
EVAL_CFG = __import__("yaml").safe_load((CFG_DIR / "evaluation.yaml").read_text())
SEED = 6304
# Smoke mode (env TASK3_SMOKE=1): 2 epochs x 5 updates per trained run, outputs under _smoke/, and notebook 03 uses
# RANDOM stand-in images and labels, so no Sketch image or label is touched.
SMOKE = os.environ.get("TASK3_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T3 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T3 / "results" / "tables"      # protocol/data-prep files from notebook 00 (same in smoke and real mode)
DATA_TAB.mkdir(parents=True, exist_ok=True)
CKPT = T3 / "checkpoints" / SUB            # git-ignored
CACHE = T3 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ERM_DIR = REPO / "task2" / "checkpoints" / "source_only"      # ERM = Task 2 Source-only, loaded, never retrained
TRAIN_RUNS = ["dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
MAIN_RUNS = ["erm", "dan_dg", "sam"]
STUDY_RUNS = ["dan_dg_lambda0.1", "dan_dg", "dan_dg_lambda10"]    # controlled study: lambda_DG in {0.1, 1, 10}
ALL_RUNS = ["erm", "dan_dg", "sam", "dan_dg_lambda0.1", "dan_dg_lambda10"]
RUN_NAME = {c: load_config(CFG_DIR, c)["run_name"] for c in ALL_RUNS}
LABEL = {"erm": "ERM", "dan_dg": "DAN-DG (λ=1)", "sam": "SAM (ρ=0.05)",
         "dan_dg_lambda0.1": "DAN-DG (λ=0.1)", "dan_dg_lambda10": "DAN-DG (λ=10)"}


def run_dir(n):
    return ERM_DIR if n == "erm" else CKPT / RUN_NAME[n]


def load_trained(n):
    """Model with the selected checkpoint of run ``n`` (ERM: the Task 2 Source-only checkpoint)."""
    from shared.models import build_model
    ck = torch.load(run_dir(n) / "best.pt", map_location="cpu", weights_only=False)
    model = build_model(7, SEED)
    model.load_state_dict(ck["model"])
    return model.to(DEVICE).eval(), ck


plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
from shared.models import build_model
from shared.training import train
from task3.methods import build_method

source = proto.load_source_data()
print({d: len(source["train"][d][1]) for d in source["train"]}, "| val:", {d: len(source["val"][d][1]) for d in source["val"]})

{'photo': 1336, 'art_painting': 1638, 'cartoon': 1875} | val: {'photo': 334, 'art_painting': 410, 'cartoon': 469}


In [3]:
runs = os.environ.get("TASK3_RUNS", ",".join(TRAIN_RUNS)).split(",")
SUMMARIES = {}
for name in runs:
    cfg = load_config(CFG_DIR, name)
    if SMOKE:
        cfg["train"]["max_epochs"] = 2
    rd = CKPT / cfg["run_name"]
    if (rd / "summary.json").exists():
        print(f"skip {name}: finished run already in {rd}")
        SUMMARIES[name] = json.loads((rd / "summary.json").read_text())
        continue
    print(f"\n=== {name} ({cfg['method']}) ===")
    model = build_model(cfg["model"]["num_classes"], cfg["seed"])
    method = build_method(cfg)
    assert not method.uses_target
    SUMMARIES[name] = train(model, method, source, cfg, rd, DEVICE, target_images=None,
                            iters_override=5 if SMOKE else None)
    del model, method
    torch.cuda.empty_cache()


=== dan_dg (dan_dg) ===


[dan_dg_lambda1] epoch  1  cls_loss=1.9320  mmd=0.5678  total_loss=2.4998  train_acc=0.2078  mmd_photo_art_painting=0.5119  mmd_photo_cartoon=0.5934  mmd_art_painting_cartoon=0.5981  val_mean_f1=0.0507  *  (15s)


[dan_dg_lambda1] epoch  2  cls_loss=1.9395  mmd=0.5122  total_loss=2.4517  train_acc=0.2163  mmd_photo_art_painting=0.4746  mmd_photo_cartoon=0.5446  mmd_art_painting_cartoon=0.5175  val_mean_f1=0.0507  (15s)


[dan_dg_lambda1] epoch  3  cls_loss=1.9742  mmd=0.5214  total_loss=2.4956  train_acc=0.2105  mmd_photo_art_painting=0.4760  mmd_photo_cartoon=0.5727  mmd_art_painting_cartoon=0.5154  val_mean_f1=0.0507  (15s)


[dan_dg_lambda1] epoch  4  cls_loss=1.9396  mmd=0.5006  total_loss=2.4402  train_acc=0.2113  mmd_photo_art_painting=0.4757  mmd_photo_cartoon=0.5080  mmd_art_painting_cartoon=0.5181  val_mean_f1=0.0507  (15s)


...

[dan_dg_lambda10] epoch  2  cls_loss=1.9400  mmd=0.4754  total_loss=6.6943  train_acc=0.1979  mmd_photo_art_painting=0.4078  mmd_photo_cartoon=0.5274  mmd_art_painting_cartoon=0.4911  val_mean_f1=0.0361  (17s)


[dan_dg_lambda10] epoch  3  cls_loss=1.9316  mmd=0.4777  total_loss=6.7090  train_acc=0.1906  mmd_photo_art_painting=0.4817  mmd_photo_cartoon=0.4941  mmd_art_painting_cartoon=0.4574  val_mean_f1=0.0507  (17s)


[dan_dg_lambda10] epoch  4  cls_loss=1.9323  mmd=0.4726  total_loss=6.6579  train_acc=0.2145  mmd_photo_art_painting=0.4903  mmd_photo_cartoon=0.4439  mmd_art_painting_cartoon=0.4835  val_mean_f1=0.0507  (17s)


[dan_dg_lambda10] epoch  5  cls_loss=1.9317  mmd=0.4903  total_loss=6.8346  train_acc=0.2156  mmd_photo_art_painting=0.4467  mmd_photo_cartoon=0.5131  mmd_art_painting_cartoon=0.5111  val_mean_f1=0.0364  (17s)


[dan_dg_lambda10] epoch  6  cls_loss=1.9241  mmd=0.4779  total_loss=6.7028  train_acc=0.2051  mmd_photo_art_painting=0.4513  mmd_photo_cartoon=0.5080  mmd_art_painting_cartoon=0.4743  val_mean_f1=0.0507  (17s)


In [4]:
erm = json.loads((ERM_DIR / "summary.json").read_text())
rows = [{"config": "erm", "run": "task2/source_only (reused, not retrained)", "method": "erm", "best_epoch": erm["best_epoch"],
         "epochs_run": erm["epochs_run"], "stopped_early": erm["stopped_early"], "best_mean_val_macro_f1": erm["best_mean_val_macro_f1"],
         "train_minutes": erm["train_seconds"] / 60}]
for n in TRAIN_RUNS:
    p = CKPT / RUN_NAME[n] / "summary.json"
    if p.exists():
        s = json.loads(p.read_text())
        rows.append({"config": n, "run": s["run_name"], "method": s["method"], "best_epoch": s["best_epoch"], "epochs_run": s["epochs_run"],
                     "stopped_early": s["stopped_early"], "best_mean_val_macro_f1": s["best_mean_val_macro_f1"], "train_minutes": s["train_seconds"] / 60})
TRAIN_SUMMARY = pd.DataFrame(rows)
TRAIN_SUMMARY.to_csv(TAB / "task3_training_summary.csv", index=False)
TRAIN_SUMMARY.round(4)

,config,run,method,best_epoch,epochs_run,stopped_early,best_mean_val_macro_f1,train_minutes
0,erm,"task2/source_only (reused, not retrained)",erm,6,11,True,0.9317,2.2367
1,dan_dg,dan_dg_lambda1,dan_dg,1,6,True,0.0507,1.5365
2,sam,sam_rho0.05,sam,5,10,True,0.9578,3.8173
3,dan_dg_lambda0.1,dan_dg_lambda0.1,dan_dg,5,10,True,0.9542,2.7960
4,dan_dg_lambda10,dan_dg_lambda10,dan_dg,1,6,True,0.0507,1.6730
